# Strategy - Regime Momentum

In [ ]:
# ============================================================
# S02 - Regime Aware Momentum
# ============================================================

# ============================================================
# Imports
# ============================================================

import xarray as xr
import numpy as np

import qnt.data as qndata
import qnt.ta as qnta
import qnt.output as qnout

# ============================================================
# Load Data
# ============================================================

data = qndata.cryptodaily_load_data(
    min_date="2015-01-01"
)

# ============================================================
# Strategy
# ============================================================

def strategy(data):

    # --------------------------------------------------------
    # Extract Fields
    # --------------------------------------------------------

    close = data.sel(field="close")
    is_liquid = data.sel(field="is_liquid")

    # --------------------------------------------------------
    # BTC Market Regime Filter
    # --------------------------------------------------------

    btc = close.sel(asset="BTC")

    btc_sma200 = qnta.sma(
        btc,
        200
    )

    bull_market = xr.where(
        btc > btc_sma200,
        1,
        0
    )

    # --------------------------------------------------------
    # Daily Returns
    # --------------------------------------------------------

    ret = close / close.shift(time=1) - 1

    # --------------------------------------------------------
    # Volatility Estimate
    # --------------------------------------------------------

    vol20 = qnta.std(
        ret,
        20
    )

    # --------------------------------------------------------
    # Momentum Signals
    # --------------------------------------------------------

    mom21 = close / close.shift(time=21) - 1

    mom63 = close / close.shift(time=63) - 1

    # --------------------------------------------------------
    # Risk Adjusted Momentum
    # --------------------------------------------------------

    risk_adj_mom = (
        mom63 /
        (vol20 + 1e-6)
    )

    # --------------------------------------------------------
    # Drawdown Quality
    # --------------------------------------------------------

    rolling_high = close.rolling(
        time=90
    ).max()

    drawdown_factor = (
        close /
        (rolling_high + 1e-6)
    )

    # --------------------------------------------------------
    # Composite Score
    # --------------------------------------------------------

    score = (
        0.45 * mom21.rank("asset").fillna(0)
        +
        0.35 * risk_adj_mom.rank("asset").fillna(0)
        +
        0.20 * drawdown_factor.rank("asset").fillna(0)
    )

    score = score * is_liquid

    # --------------------------------------------------------
    # Select Strongest Assets
    # --------------------------------------------------------

    ranks = score.rank("asset")

    top_assets = xr.where(
        ranks >= 6,
        1,
        0
    )

    weights = score * top_assets

    # --------------------------------------------------------
    # Inverse Volatility Weighting
    # --------------------------------------------------------

    inv_vol = 1 / (vol20 + 1e-6)

    weights = weights * inv_vol

    # --------------------------------------------------------
    # Normalize Portfolio
    # --------------------------------------------------------

    denom = weights.sum("asset")

    weights = xr.where(
        denom > 0,
        weights / denom,
        0
    )

    # --------------------------------------------------------
    # Apply Regime Filter
    # --------------------------------------------------------

    weights = weights * bull_market

    return weights.fillna(0)

# ============================================================
# Generate Portfolio Weights
# ============================================================

weights = strategy(data)

# ============================================================
# Clean Weights
# ============================================================

weights = qnout.clean(
    weights,
    data,
    "crypto_daily_long"
)

# ============================================================
# Write Submission Output
# ============================================================

qnout.write(weights)

100% (15259192 of 15259192) |############| Elapsed Time: 0:00:00 Time:  0:00:000:00
Output cleaning...
Fix unique timestamps
Forward filling missing prices...
Check liquidity...
Ok.
Check for missed dates...
Ok.
Check positive positions...
Ok.
Final normalization...
Output cleaning complete.
Write output: /root/fractions.nc.gz


In [ ]:
# ============================================================
# Strategy Statistics
# ============================================================

import qnt.stats as qnstats

stats = qnstats.calc_stat(
    data,
    weights.sel(time=slice("2016-01-01", None))
)

stats_pd = stats.to_pandas()

print(stats_pd.tail())

print("\nFinal Metrics:")

print(
    stats_pd.iloc[-1][[
        "equity",
        "sharpe_ratio",
        "max_drawdown",
        "avg_turnover",
        "avg_holding_time"
    ]]
)

field            equity  relative_return  volatility  underwater  \
time                                                               
2026-06-05  2209.317633              0.0    0.608425   -0.393563   
2026-06-06  2209.317633              0.0    0.608345   -0.393563   
2026-06-07  2209.317633              0.0    0.608266   -0.393563   
2026-06-08  2209.317633              0.0    0.608187   -0.393563   
2026-06-09  2209.317633              0.0    0.608107   -0.393563   

field       max_drawdown  sharpe_ratio  mean_return  bias  instruments  \
time                                                                     
2026-06-05     -0.869241      1.520313     0.924996   0.0         34.0   
2026-06-06     -0.869241      1.520112     0.924753   0.0         34.0   
2026-06-07     -0.869241      1.519911     0.924510   0.0         34.0   
2026-06-08     -0.869241      1.519711     0.924268   0.0         34.0   
2026-06-09     -0.869241      1.519510     0.924025   0.0         34.0   

field       avg_turnover  avg_holding_time  
time                                        
2026-06-05      0.091489         14.244739  
2026-06-06      0.091465         14.244739  
2026-06-07      0.091441         14.244739  
2026-06-08      0.091417         14.244739  
2026-06-09      0.091393         14.244739  

Final Metrics:
field
equity              2209.317633
sharpe_ratio           1.519510
max_drawdown          -0.869241
avg_turnover           0.091393
avg_holding_time      14.244739
Name: 2026-06-09 00:00:00, dtype: float64